# 62 — Spreadsheet Analysis Agent

## Goal
Build an agent that can **load CSV/XLSX files**, answer questions
about the data, **generate charts**, and **explain anomalies**.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Code-generating agent** | LLM writes and executes Python/pandas code |
| **PythonREPLTool** | Safe sandbox for agent-written code |
| **Data analysis tools** | Custom tools for loading, summarizing, charting |
| **Natural language → code** | User asks in English, agent writes code |

## Architecture
```
User Question → Agent (qwen3.5) → [Choose tool]
                                       ↓
                     load_csv / summarize_data / run_python / create_chart
                                       ↓
                               Observe results → Answer
```

## Stack
- `LangGraph` + `ChatOllama` + custom data tools
- `pandas` + `matplotlib` for data analysis
- `openpyxl` for XLSX support

In [ ]:
import os, warnings, json, io
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for agent
import matplotlib.pyplot as plt

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports OK")

## Step 1 — Create Sample Data

We'll create a realistic sales dataset for the agent to analyze.

In [ ]:
import numpy as np
np.random.seed(42)

n = 200
regions = np.random.choice(["North", "South", "East", "West"], n)
products = np.random.choice(["Widget A", "Widget B", "Widget C", "Premium X"], n)
months = np.random.choice(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                            "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"], n)
units = np.random.randint(10, 500, n)
price = np.where(np.isin(products, ["Premium X"]),
                 np.random.uniform(80, 150, n),
                 np.random.uniform(10, 50, n))
revenue = units * price

# Inject some anomalies
revenue[42] = 999999.99  # Suspiciously high
units[100] = 0           # Zero units but has revenue (data error)
revenue[150] = -500.0    # Negative revenue (return?)

df = pd.DataFrame({
    "region": regions, "product": products, "month": months,
    "units_sold": units, "unit_price": price.round(2), "revenue": revenue.round(2),
})

DATA_PATH = "sample_sales.csv"
df.to_csv(DATA_PATH, index=False)
print(f"Created {DATA_PATH}: {len(df)} rows")
df.head()

## Step 2 — Define Data Analysis Tools

Each tool gives the agent a specific capability.
The agent decides which to call based on the question.

In [ ]:
# Global dataframe store
_loaded_df = {}

@tool
def load_spreadsheet(file_path: str) -> str:
    """Load a CSV or XLSX file. Returns column names, dtypes, shape, and first 5 rows."""
    try:
        if file_path.endswith(".xlsx"):
            data = pd.read_excel(file_path)
        else:
            data = pd.read_csv(file_path)
        _loaded_df["current"] = data
        info = []
        info.append(f"Shape: {data.shape[0]} rows × {data.shape[1]} columns")
        info.append(f"Columns: {list(data.columns)}")
        info.append(f"Dtypes:\n{data.dtypes.to_string()}")
        info.append(f"First 5 rows:\n{data.head().to_string()}")
        return "\n".join(info)
    except Exception as e:
        return f"Error loading file: {e}"

@tool
def summarize_data() -> str:
    """Generate descriptive statistics of the loaded spreadsheet. Includes mean, std, min, max, and null counts."""
    if "current" not in _loaded_df:
        return "No spreadsheet loaded. Use load_spreadsheet first."
    data = _loaded_df["current"]
    info = []
    info.append(f"Numeric summary:\n{data.describe().to_string()}")
    nulls = data.isnull().sum()
    if nulls.any():
        info.append(f"Null values:\n{nulls[nulls > 0].to_string()}")
    else:
        info.append("No null values found.")
    return "\n".join(info)

@tool
def query_data(pandas_query: str) -> str:
    """Run a pandas query/expression on the loaded data. Example: 'df[df.revenue > 10000].head(10)' or 'df.groupby("region").revenue.sum()'. The dataframe is available as 'df'."""
    if "current" not in _loaded_df:
        return "No spreadsheet loaded. Use load_spreadsheet first."
    df = _loaded_df["current"]
    try:
        result = eval(pandas_query, {"df": df, "pd": pd, "np": np})
        if isinstance(result, pd.DataFrame):
            return result.to_string(max_rows=20)
        elif isinstance(result, pd.Series):
            return result.to_string()
        return str(result)
    except Exception as e:
        return f"Query error: {e}"

@tool
def create_chart(chart_type: str, x_column: str, y_column: str, title: str) -> str:
    """Create a chart from the loaded data. chart_type: 'bar', 'line', 'scatter', 'pie', 'hist'. Saves chart to 'chart_output.png'."""
    if "current" not in _loaded_df:
        return "No spreadsheet loaded. Use load_spreadsheet first."
    df = _loaded_df["current"]
    try:
        fig, ax = plt.subplots(figsize=(10, 6))
        if chart_type == "bar":
            plot_data = df.groupby(x_column)[y_column].sum()
            plot_data.plot(kind="bar", ax=ax)
        elif chart_type == "line":
            plot_data = df.groupby(x_column)[y_column].sum()
            plot_data.plot(kind="line", ax=ax, marker="o")
        elif chart_type == "scatter":
            ax.scatter(df[x_column], df[y_column], alpha=0.5)
        elif chart_type == "pie":
            plot_data = df.groupby(x_column)[y_column].sum()
            plot_data.plot(kind="pie", ax=ax, autopct="%1.1f%%")
        elif chart_type == "hist":
            df[y_column].hist(ax=ax, bins=20)
        ax.set_title(title)
        plt.tight_layout()
        plt.savefig("chart_output.png", dpi=100)
        plt.close()
        return f"Chart saved to chart_output.png. Type: {chart_type}, X: {x_column}, Y: {y_column}"
    except Exception as e:
        return f"Chart error: {e}"

@tool
def detect_anomalies() -> str:
    """Detect anomalies in numeric columns using IQR method. Returns rows with values outside 1.5*IQR."""
    if "current" not in _loaded_df:
        return "No spreadsheet loaded. Use load_spreadsheet first."
    df = _loaded_df["current"]
    anomalies = []
    for col in df.select_dtypes(include=[np.number]).columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower) | (df[col] > upper)]
        if len(outliers) > 0:
            anomalies.append(f"Column '{col}': {len(outliers)} anomalies (range: {lower:.2f} to {upper:.2f})")
            anomalies.append(outliers[[col]].head(5).to_string())
    return "\n".join(anomalies) if anomalies else "No anomalies detected."

tools = [load_spreadsheet, summarize_data, query_data, create_chart, detect_anomalies]
print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

## Step 3 — Create the Spreadsheet Agent

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

SYSTEM_PROMPT = """You are a data analyst assistant. You help users analyze spreadsheets.

Workflow:
1. First load the spreadsheet using load_spreadsheet
2. Summarize the data to understand its structure  
3. Answer questions using query_data for specific analysis
4. Create charts when visual representation helps
5. Check for anomalies when asked about data quality

Always explain your findings in plain English after getting tool results."""

spreadsheet_agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=SYSTEM_PROMPT,
)

def ask_data_agent(question: str):
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")
    result = spreadsheet_agent.invoke({"messages": [{"role": "user", "content": question}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL: {tc['name']}({str(tc['args'])[:100]})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:300]}...")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 ANSWER:\n{msg.content}")
    return result

print("Spreadsheet agent ready!")

In [ ]:
# Question 1: Load and explore
ask_data_agent("Load sample_sales.csv and give me an overview of the data.")

In [ ]:
# Question 2: Specific analysis
ask_data_agent("Which region has the highest total revenue? Show me a bar chart of revenue by region.")

In [ ]:
# Display the generated chart
if os.path.exists("chart_output.png"):
    from IPython.display import Image, display
    display(Image("chart_output.png"))

In [ ]:
# Question 3: Anomaly detection
ask_data_agent("Are there any anomalies or data quality issues in this dataset? Explain what you find.")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Multiple tools** | Agent chooses from 5 tools based on question |
| **Stateful tools** | `_loaded_df` dict persists data across tool calls |
| **Safe code execution** | `eval()` with limited namespace (df, pd, np only) |
| **Anomaly detection** | IQR method finds statistical outliers |
| **Chart generation** | Agent creates visualizations and saves to disk |

## Tool Selection Pattern
```
"Load the file"         → load_spreadsheet
"What's the average?"   → query_data (writes pandas code)
"Show me a chart"       → create_chart
"Any data issues?"      → detect_anomalies
"Give me an overview"   → summarize_data
```